# YOLOv8n — Road Detection (13 Classes)

| | |
|---|---|
| **Model** | YOLOv8n (Nano) |
| **Resolution** | 640×640 |
| **GPU** | NVIDIA GTX 1650 / 4 GB VRAM |
| **Classes** | 13 |
| **Output** | best.pt |

Run cells **in order**.

---


## Cell 1 — GPU & package check


In [ ]:
import sys
import torch
import ultralytics

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Ultralytics : {ultralytics.__version__}")
print()

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1e9
    print(f"GPU  : {gpu.name}")
    print(f"VRAM : {vram:.1f} GB")
    print(f"CUDA : {torch.version.cuda}")
    DEVICE = 0
    if vram < 5.0:
        print("\n⚠️  VRAM < 5 GB — batch=8 and cache=False will be used to avoid OOM.")
else:
    print("WARNING: No CUDA GPU found — training will be very slow on CPU.")
    DEVICE = "cpu"

print(f"\nTraining device : {DEVICE}")


## Cell 2 — Paths & constants


In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("dataset")        # folder with images/, labels/ and data.yaml
YAML_PATH    = DATASET_ROOT / "data.yaml"
RUNS_DIR     = Path("runs")
RUN_NAME     = "detection_routiere_v1"

# ── Class map ─────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0:  "chaussee_degradee",
    1:  "ralentisseur",
    2:  "cedez_le_passage",
    3:  "passage_pietons",
    4:  "stop",
    5:  "succession_de_virages",
    6:  "feu_rouge",
    7:  "feu_vert",
    8:  "vitesse_100",
    9:  "vitesse_30",
    10: "vitesse_40",
    11: "vitesse_60",
    12: "vitesse_80",
}

# ── Colors BGR (OpenCV) ───────────────────────────────────────────────────
CLASS_COLORS = {
    0:  (80,  80,  220),
    1:  (220, 130,  50),
    2:  (50,  200,  50),
    3:  (200, 200,  50),
    4:  (50,  50,  220),
    5:  (180,  50, 180),
    6:  (50,   50, 255),
    7:  (50,  220,  50),
    8:  (255, 180,   0),
    9:  (0,   200, 255),
    10: (100, 100, 255),
    11: (255,  80, 120),
    12: (80,  255, 200),
}

# ── Hyperparameters (GTX 1650 — 4 GB VRAM) ───────────────────────────────
TRAIN_IMGSZ = 640
BATCH_SIZE  = 8      # safe for 4 GB VRAM with YOLOv8n
EPOCHS      = 200
PATIENCE    = 30     # early stopping
# FIX: workers=0 on Windows avoids DataLoader multiprocessing spawn issues.
# Increase to 2-4 only if you are on Linux or if you are sure your setup supports it.
WORKERS     = 0

# ── Path checks ───────────────────────────────────────────────────────────
for p in [DATASET_ROOT, YAML_PATH]:
    status = "OK" if p.exists() else "NOT FOUND ⚠️"
    print(f"[{status}] {p}")

RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Runs dir : {RUNS_DIR}")
print(f"\n{len(CLASS_NAMES)} classes configured.")


## Cell 3 — Dataset integrity check (train + val)


In [ ]:
all_issues = []

for split in ["train", "val"]:
    img_dir = DATASET_ROOT / "images" / split
    lbl_dir = DATASET_ROOT / "labels" / split

    images = sorted(
        list(img_dir.glob("*.jpg"))  +
        list(img_dir.glob("*.jpeg")) +
        list(img_dir.glob("*.png"))
    )

    missing = empty = bad_format = bad_class = bad_coords = 0

    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + ".txt")

        if not lbl_path.exists():
            missing += 1
            all_issues.append(f"[{split}] Missing label: {lbl_path.name}")
            continue

        lines = [l.strip() for l in lbl_path.read_text(encoding="utf-8").splitlines() if l.strip()]

        if not lines:
            empty += 1
            continue

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                bad_format += 1
                all_issues.append(f"[{split}] Bad format in {lbl_path.name}: '{line}'")
                continue
            try:
                cls    = int(parts[0])
                coords = list(map(float, parts[1:]))
            except ValueError:
                bad_format += 1
                all_issues.append(f"[{split}] Non-numeric value in {lbl_path.name}: '{line}'")
                continue
            if cls not in CLASS_NAMES:
                bad_class += 1
                all_issues.append(f"[{split}] Unknown class {cls} in {lbl_path.name}")
            if not all(0.0 <= v <= 1.0 for v in coords):
                bad_coords += 1
                all_issues.append(f"[{split}] Out-of-range coords in {lbl_path.name}: {coords}")

    print(f"[{split:5s}]  images={len(images)}  "
          f"missing={missing}  empty={empty}  "
          f"bad_format={bad_format}  bad_class={bad_class}  bad_coords={bad_coords}")

print()
if all_issues:
    print(f"*** {len(all_issues)} issue(s) found — fix before training ***")
    for issue in all_issues[:30]:
        print(" ", issue)
    if len(all_issues) > 30:
        print(f"  ... and {len(all_issues) - 30} more")
else:
    print("✅ Dataset clean. Ready for training.")


## Cell 4 — Class distribution (train + val)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from collections import defaultdict

# ── Count annotations & images per split ──────────────────────────────────
stats      = {}
img_counts = {}

for split in ["train", "val", "test"]:
    lbl_dir = DATASET_ROOT / "labels" / split
    img_dir = DATASET_ROOT / "images" / split
    counts  = defaultdict(int)

    if lbl_dir.exists():
        for lbl_path in lbl_dir.glob("*.txt"):
            for line in lbl_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) == 5:
                    cls = int(parts[0])
                    if cls in CLASS_NAMES:
                        counts[cls] += 1

    stats[split] = counts
    img_counts[split] = (
        len(list(img_dir.glob("*.jpg")))  +
        len(list(img_dir.glob("*.jpeg"))) +
        len(list(img_dir.glob("*.png")))
        if img_dir.exists() else 0
    )

# ── Build DataFrame ────────────────────────────────────────────────────────
rows = []
for cid, name in CLASS_NAMES.items():
    tr = stats["train"][cid]
    va = stats["val"][cid]
    te = stats["test"][cid]
    tot = tr + va + te
    rows.append({
        "id": cid, "class": name,
        "train": tr, "val": va, "test": te, "total": tot,
        "train_%": round(tr / tot * 100, 1) if tot else 0.0,
        "val_%":   round(va / tot * 100, 1) if tot else 0.0,
        "test_%":  round(te / tot * 100, 1) if tot else 0.0,
    })

df = pd.DataFrame(rows).set_index("id")

# ── Print table ────────────────────────────────────────────────────────────
SEP = "=" * 85
sep = "-" * 85
print(SEP)
print("  ANNOTATION DISTRIBUTION PER CLASS")
print(SEP)
print(f"{'ID':<4} {'Class':<25} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>7}  "
      f"{'Train%':>7} {'Val%':>6} {'Test%':>6}")
print(sep)
for idx, row in df.iterrows():
    print(f"{idx:<4} {row['class']:<25} {int(row['train']):>7} {int(row['val']):>7} "
          f"{int(row['test']):>7} {int(row['total']):>7}  "
          f"{row['train_%']:>6.1f}% {row['val_%']:>5.1f}% {row['test_%']:>5.1f}%")
print(SEP)
tot_tr = df["train"].sum(); tot_va = df["val"].sum()
tot_te = df["test"].sum(); grand  = df["total"].sum()
print(f"{'':4} {'TOTAL':<25} {int(tot_tr):>7} {int(tot_va):>7} "
      f"{int(tot_te):>7} {int(grand):>7}  "
      f"{tot_tr/grand*100:>6.1f}% {tot_va/grand*100:>5.1f}% {tot_te/grand*100:>5.1f}%")
print(SEP)

# ── Image counts ───────────────────────────────────────────────────────────
total_imgs = sum(img_counts.values())
print()
print("  IMAGE COUNT PER SPLIT")
print(sep)
for split in ["train", "val", "test"]:
    n   = img_counts[split]
    pct = 100 * n / total_imgs if total_imgs else 0
    bar = "█" * int(pct / 2)
    print(f"  {split:5s} : {n:5d} images  ({pct:5.1f}%)  {bar}")
print(f"  {'TOTAL':5s} : {total_imgs:5d} images")

# ── Shared style helper ────────────────────────────────────────────────────
BG   = "#0f1117"
AX   = "#1a1d27"
GRID = "#2a2d3a"
COLORS = {"train": "#4ade80", "val": "#60a5fa", "test": "#f87171"}

def style_ax(ax, title):
    ax.set_facecolor(AX)
    ax.set_title(title, color="white", fontsize=13, pad=12)
    ax.tick_params(colors="white")
    ax.grid(axis="y", color=GRID, linestyle="--", alpha=0.6)
    for spine in ax.spines.values():
        spine.set_color(GRID)

short_names = [
    "Chaussée\ndégradée", "Ralenti-\nsseur", "Cédez\npassage",
    "Passage\npiétons", "STOP", "Succession\nvirages",
    "Feu rouge", "Feu vert", "Vit. 100",
    "Vit. 30", "Vit. 40", "Vit. 60", "Vit. 80"
]
x = np.arange(len(CLASS_NAMES))

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Stacked bar: total annotations per class
# ════════════════════════════════════════════════════════════════════════════
fig1, ax = plt.subplots(figsize=(18, 6))
fig1.patch.set_facecolor(BG)
style_ax(ax, "Annotation count per class — stacked by split")
ax.set_ylabel("Annotations", color="white")

bottom = np.zeros(len(CLASS_NAMES))
for split in ["train", "val", "test"]:
    vals = np.array([stats[split][i] for i in range(len(CLASS_NAMES))], dtype=float)
    bars = ax.bar(x, vals, bottom=bottom, label=split.capitalize(),
                  color=COLORS[split], alpha=0.9)
    for rect, v, b in zip(bars, vals, bottom):
        if v > 20:
            ax.text(rect.get_x() + rect.get_width()/2, b + v/2,
                    str(int(v)), ha="center", va="center",
                    color="white", fontsize=7, fontweight="bold")
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(short_names, color="white", fontsize=8)
ax.legend(facecolor=AX, labelcolor="white")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
p1 = RUNS_DIR / "dist_1_stacked.png"
plt.savefig(str(p1), dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"\n📊 Saved: {p1}")

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Grouped bar: train / val / test side by side
# ════════════════════════════════════════════════════════════════════════════
fig2, ax = plt.subplots(figsize=(18, 6))
fig2.patch.set_facecolor(BG)
style_ax(ax, "Annotation count per class — grouped by split")
ax.set_ylabel("Annotations", color="white")

w = 0.26
for k, split in enumerate(["train", "val", "test"]):
    vals = [stats[split][i] for i in range(len(CLASS_NAMES))]
    bars = ax.bar(x + (k - 1) * w, vals, w,
                  label=split.capitalize(), color=COLORS[split], alpha=0.9)
    for rect, v in zip(bars, vals):
        if v > 0:
            ax.text(rect.get_x() + rect.get_width()/2, v + 8,
                    str(int(v)), ha="center", va="bottom",
                    color="white", fontsize=6)

ax.set_xticks(x)
ax.set_xticklabels(short_names, color="white", fontsize=8)
ax.legend(facecolor=AX, labelcolor="white")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
p2 = RUNS_DIR / "dist_2_grouped.png"
plt.savefig(str(p2), dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"📊 Saved: {p2}")

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Horizontal bar: annotation % per class (train vs val vs test)
# ════════════════════════════════════════════════════════════════════════════
fig3, ax = plt.subplots(figsize=(12, 8))
fig3.patch.set_facecolor(BG)
ax.set_facecolor(AX)
ax.set_title("Split % per class (annotations)", color="white", fontsize=13, pad=12)
ax.set_xlabel("Percentage (%)", color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color(GRID)

class_labels = [f"{i}: {CLASS_NAMES[i]}" for i in range(len(CLASS_NAMES))]
y = np.arange(len(CLASS_NAMES))
left = np.zeros(len(CLASS_NAMES))

for split in ["train", "val", "test"]:
    pcts = np.array([
        stats[split][i] / (stats["train"][i] + stats["val"][i] + stats["test"][i]) * 100
        if (stats["train"][i] + stats["val"][i] + stats["test"][i]) > 0 else 0
        for i in range(len(CLASS_NAMES))
    ])
    bars = ax.barh(y, pcts, left=left, label=split.capitalize(),
                   color=COLORS[split], alpha=0.9)
    for i, (rect, pct) in enumerate(zip(bars, pcts)):
        if pct > 4:
            ax.text(left[i] + pct/2,
                    rect.get_y() + rect.get_height()/2,
                    f"{pct:.1f}%", ha="center", va="center",
                    color="white", fontsize=8, fontweight="bold")
    left += pcts

ax.set_yticks(y)
ax.set_yticklabels(class_labels, color="white", fontsize=9)
ax.set_xlim(0, 100)
ax.axvline(65, color="#facc15", linestyle="--", lw=1, alpha=0.5)
ax.axvline(80, color="#facc15", linestyle="--", lw=1, alpha=0.5)
ax.legend(facecolor=AX, labelcolor="white")
ax.grid(axis="x", color=GRID, linestyle="--", alpha=0.5)
plt.tight_layout()
p3 = RUNS_DIR / "dist_3_pct_per_class.png"
plt.savefig(str(p3), dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"📊 Saved: {p3}")

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Pie chart: image split proportions
# ════════════════════════════════════════════════════════════════════════════
fig4, ax = plt.subplots(figsize=(7, 7))
fig4.patch.set_facecolor(BG)
ax.set_facecolor(AX)
ax.set_title("Image split proportion", color="white", fontsize=13, pad=12)

pie_sizes  = [img_counts[s] for s in ["train", "val", "test"]]
pie_labels = [
    f"{s.capitalize()}\n{img_counts[s]:,} imgs\n({100*img_counts[s]/total_imgs:.1f}%)"
    if total_imgs else s
    for s in ["train", "val", "test"]
]
pie_colors = [COLORS[s] for s in ["train", "val", "test"]]
wedges, texts = ax.pie(
    pie_sizes, labels=pie_labels, colors=pie_colors,
    startangle=140,
    textprops={"color": "white", "fontsize": 11},
    wedgeprops={"linewidth": 2, "edgecolor": BG},
)
plt.tight_layout()
p4 = RUNS_DIR / "dist_4_image_split_pie.png"
plt.savefig(str(p4), dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"📊 Saved: {p4}")

print("\n✅ All 4 distribution charts saved.")


## Cell 5 — Random dataset preview (3×3 grid)


In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

def draw_boxes(img_path, lbl_path, class_names, class_colors):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)
            color = class_colors.get(cls, (255, 255, 0))
            color_rgb = (color[2], color[1], color[0])  # BGR → RGB
            cv2.rectangle(img, (x1, y1), (x2, y2), color_rgb, 2)
            label = class_names.get(cls, f"cls{cls}")
            cv2.putText(img, label, (x1, max(y1 - 5, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_rgb, 1)
    return img

train_imgs = (
    list((DATASET_ROOT / "images" / "train").glob("*.jpg")) +
    list((DATASET_ROOT / "images" / "train").glob("*.jpeg")) +
    list((DATASET_ROOT / "images" / "train").glob("*.png"))
)

sample = random.sample(train_imgs, min(9, len(train_imgs)))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.patch.set_facecolor("#0f1117")
fig.suptitle("Dataset preview — Train (ground truth annotations)",
             color="white", fontsize=14, y=1.01)

for ax, img_path in zip(axes.flat, sample):
    lbl_path = DATASET_ROOT / "labels" / "train" / (img_path.stem + ".txt")
    img = draw_boxes(img_path, lbl_path, CLASS_NAMES, CLASS_COLORS)
    if img is not None:
        ax.imshow(img)
    ax.set_title(img_path.name[:30], color="white", fontsize=7)
    ax.axis("off")
    ax.set_facecolor("#0f1117")

plt.tight_layout()
plt.savefig(str(RUNS_DIR / "apercu_dataset.png"), dpi=120,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("🖼️  Preview saved.")


## Cell 6 — Train YOLOv8n

> **GTX 1650 note**: `batch=8`, `cache=False`, `workers=0` to stay within 4 GB VRAM.  
> Estimated time: ~2–3 h.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # auto-downloaded if absent

results = model.train(
    # ── Dataset ────────────────────────────────────────────────────────────
    data      = str(YAML_PATH),
    imgsz     = TRAIN_IMGSZ,

    # ── Training ───────────────────────────────────────────────────────────
    epochs    = EPOCHS,
    patience  = PATIENCE,
    batch     = BATCH_SIZE,
    device    = DEVICE,
    workers   = WORKERS,     # FIX: 0 avoids Windows DataLoader spawn issues
    cache     = False,        # disabled to save RAM/VRAM

    # ── Optimizer ──────────────────────────────────────────────────────────
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs   = 5,
    warmup_momentum = 0.8,

    # ── Augmentation ───────────────────────────────────────────────────────
    mosaic      = 1.0,
    mixup       = 0.1,
    degrees     = 10.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0,
    flipud      = 0.0,    # no vertical flip (signs are upright)
    fliplr      = 0.3,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,

    # ── Save & logs ────────────────────────────────────────────────────────
    project     = str(RUNS_DIR),
    name        = RUN_NAME,
    exist_ok    = True,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,

    # ── Detection thresholds ───────────────────────────────────────────────
    # FIX: conf is removed from train() — it is not a valid train argument
    # in recent ultralytics and causes a warning / is silently ignored.
    iou         = 0.7,
)

print("\n✅ Training complete.")
print(f"📁 Results: {RUNS_DIR / RUN_NAME}")


## Cell 7 — Training curves (loss + metrics)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = RUNS_DIR / RUN_NAME / "results.csv"

if not csv_path.exists():
    print(f"❌ File not found: {csv_path}")
    print("   Run Cell 6 first.")
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    print("Available columns:", df.columns.tolist())

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor("#0f1117")
    fig.suptitle("Training curves — YOLOv8n (13 classes)",
                 color="white", fontsize=15, y=1.02)

    plots = [
        ("train/box_loss",       "val/box_loss",             "Box Loss",           "Loss"),
        ("train/cls_loss",       "val/cls_loss",             "Cls Loss",           "Loss"),
        ("train/dfl_loss",       "val/dfl_loss",             "DFL Loss",           "Loss"),
        ("metrics/mAP50(B)",     None,                       "mAP@0.50",           "mAP"),
        ("metrics/mAP50-95(B)",  None,                       "mAP@0.50:0.95",      "mAP"),
        ("metrics/precision(B)", "metrics/recall(B)",        "Precision / Recall", "Value"),
    ]

    epochs = df["epoch"] if "epoch" in df.columns else range(len(df))

    for ax, (col1, col2, title, ylabel) in zip(axes.flat, plots):
        ax.set_facecolor("#1a1d27")
        # FIX: ax.spines[:] fails on matplotlib < 3.4 — use .values() loop
        for spine in ax.spines.values():
            spine.set_color("#444")
        ax.tick_params(colors="white")
        ax.set_title(title, color="white", fontsize=11)
        ax.set_xlabel("Epoch", color="#aaa", fontsize=9)
        ax.set_ylabel(ylabel, color="#aaa", fontsize=9)
        ax.grid(color="#333", linestyle="--", alpha=0.5)

        if col1 in df.columns:
            label1 = col1.split("/")[-1].replace("(B)", "")
            ax.plot(epochs, df[col1], color="#4ade80", lw=2, label=f"Train {label1}")

        if col2 and col2 in df.columns:
            label2 = col2.split("/")[-1].replace("(B)", "")
            ax.plot(epochs, df[col2], color="#f87171", lw=2, linestyle="--",
                    label=f"Val {label2}")

        ax.legend(facecolor="#1a1d27", labelcolor="white", fontsize=8)

    plt.tight_layout()
    save_path = RUNS_DIR / RUN_NAME / "training_curves.png"
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"📊 Curves saved: {save_path}")


## Cell 8 — Load best model & evaluate on val


In [ ]:
from ultralytics import YOLO

# FIX: declare as module-level variables so cells 9, 10, 12 can safely use them
best_model  = None
val_results = None

best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if not best_weights.exists():
    print(f"❌ best.pt not found: {best_weights}")
    print("   Make sure Cell 6 (training) completed successfully.")
else:
    best_model = YOLO(str(best_weights))
    print(f"✅ Model loaded: {best_weights}")

    val_results = best_model.val(
        data     = str(YAML_PATH),
        imgsz    = TRAIN_IMGSZ,
        batch    = BATCH_SIZE,
        device   = DEVICE,
        conf     = 0.25,
        iou      = 0.45,
        plots    = True,
        save_json = False,
        project  = str(RUNS_DIR),
        name     = RUN_NAME + "_val",
        exist_ok = True,
    )

    print(f"\nmAP@0.50      : {val_results.box.map50:.4f}")
    print(f"mAP@0.50:0.95 : {val_results.box.map:.4f}")
    print(f"Precision     : {val_results.box.mp:.4f}")
    print(f"Recall        : {val_results.box.mr:.4f}")


## Cell 9 — mAP per class


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# FIX: guard against val_results not being set (e.g. if Cell 8 was skipped)
if val_results is None:
    print("❌ val_results not available. Run Cell 8 first.")
else:
    # FIX: ultralytics changed the per-class AP attribute across versions:
    #   older  → val_results.box.ap50   (numpy array)
    #   newer  → val_results.box.maps   (numpy array, same values)
    # Try both; fall back gracefully.
    ap_per_class = None
    for attr in ("ap50", "maps"):
        candidate = getattr(val_results.box, attr, None)
        if candidate is not None and len(candidate) > 0:
            ap_per_class = candidate
            print(f"Using val_results.box.{attr} for per-class AP.")
            break

    if ap_per_class is None:
        print("⚠️  Per-class AP not available in this ultralytics version.")
    else:
        class_labels = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES))]
        n        = min(len(ap_per_class), len(class_labels))
        ap_vals  = list(ap_per_class[:n])
        labels   = class_labels[:n]

        print(f"{'Class':<25} {'mAP@50':>8}  Bar")
        print("-" * 55)
        for name, ap in zip(labels, ap_vals):
            bar  = "█" * int(ap * 40)
            flag = " ⚠️" if ap < 0.50 else ""
            print(f"{name:<25} {ap:8.4f}  {bar}{flag}")

        # ── Chart ─────────────────────────────────────────────────────────
        colors = ["#f87171" if v < 0.50 else "#4ade80" for v in ap_vals]

        fig, ax = plt.subplots(figsize=(12, 6))
        fig.patch.set_facecolor("#0f1117")
        ax.set_facecolor("#1a1d27")

        bars = ax.barh(labels, ap_vals, color=colors, alpha=0.9)
        ax.axvline(0.50, color="#facc15", linestyle="--", lw=1.5, label="Threshold 0.50")
        ax.set_xlim(0, 1.0)
        ax.set_xlabel("mAP@0.50", color="white")
        ax.set_title("mAP@0.50 per class", color="white", fontsize=13, pad=12)
        ax.tick_params(colors="white")
        # FIX: ax.spines[:] → loop over .values()
        for spine in ax.spines.values():
            spine.set_color("#444")
        ax.grid(axis="x", color="#333", linestyle="--", alpha=0.5)
        ax.legend(facecolor="#1a1d27", labelcolor="white")

        for bar, val in zip(bars, ap_vals):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", color="white", fontsize=9)

        plt.tight_layout()
        save_path = RUNS_DIR / RUN_NAME / "map_per_class.png"
        plt.savefig(str(save_path), dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        plt.show()
        print(f"\n📊 Per-class mAP chart saved: {save_path}")
        print("\n⚠️  Classes in red (mAP < 0.50): consider more data or targeted augmentation.")


## Cell 10 — Confusion matrix


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

val_dir = RUNS_DIR / (RUN_NAME + "_val")

for path, title in [
    (val_dir / "confusion_matrix_normalized.png", "Confusion matrix (normalized)"),
    (val_dir / "confusion_matrix.png",            "Confusion matrix (absolute)"),
]:
    if path.exists():
        img = mpimg.imread(str(path))
        fig, ax = plt.subplots(figsize=(14, 12))
        fig.patch.set_facecolor("#0f1117")
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=13, pad=10)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  Not found: {path}")
        print("    Make sure plots=True was set in Cell 8.")


## Cell 11 — Precision-Recall curves


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

val_dir = RUNS_DIR / (RUN_NAME + "_val")
pr_path = val_dir / "PR_curve.png"

if pr_path.exists():
    img = mpimg.imread(str(pr_path))
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor("#0f1117")
    ax.imshow(img)
    ax.set_title("Precision-Recall curves per class", color="white", fontsize=13, pad=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️  PR_curve.png not found in {val_dir}")
    print("    Run Cell 8 with plots=True first.")


## Cell 12 — GT vs predictions on val


In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

# FIX: guard against best_model not being set
if best_model is None:
    print("❌ best_model not available. Run Cell 8 first.")
else:
    val_img_dir = DATASET_ROOT / "images" / "val"
    val_lbl_dir = DATASET_ROOT / "labels" / "val"
    save_dir    = RUNS_DIR / RUN_NAME / "val_compare"
    save_dir.mkdir(parents=True, exist_ok=True)

    val_imgs = (
        list(val_img_dir.glob("*.jpg"))  +
        list(val_img_dir.glob("*.jpeg")) +
        list(val_img_dir.glob("*.png"))
    )

    sample = random.sample(val_imgs, min(9, len(val_imgs)))

    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.patch.set_facecolor("#0f1117")
    fig.suptitle("Val — 🟢 Ground truth  |  🔴 Model predictions",
                 color="white", fontsize=14, y=1.01)

    for ax, img_path in zip(axes.flat, sample):
        img = cv2.imread(str(img_path))
        if img is None:
            ax.axis("off")
            continue
        h, w = img.shape[:2]

        # Ground truth (green)
        lbl_path = val_lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 220, 0), 2)
                cv2.putText(img, f"GT:{CLASS_NAMES.get(cls, cls)}",
                            (x1, max(y1 - 5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 220, 0), 1)

        # Predictions (red)
        result = best_model.predict(
            source  = str(img_path),
            imgsz   = TRAIN_IMGSZ,
            conf    = 0.25,
            iou     = 0.45,
            device  = DEVICE,
            verbose = False,
        )[0]

        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls  = int(box.cls[0])
                conf = float(box.conf[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 220), 2)
                cv2.putText(img, f"{CLASS_NAMES.get(cls, cls)} {conf:.2f}",
                            (x1, y2 + 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 220), 1)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(img_path.name[:28], color="white", fontsize=7)
        ax.axis("off")
        ax.set_facecolor("#0f1117")
        cv2.imwrite(str(save_dir / img_path.name), img)

    plt.tight_layout()
    plt.savefig(str(RUNS_DIR / RUN_NAME / "val_comparison_grid.png"),
                dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"✅ {len(sample)} comparison images saved to {save_dir}")


## Cell 13 — Final summary


In [ ]:
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
last_weights = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

print("=" * 60)
print("🏁  TRAINING COMPLETE — SUMMARY")
print("=" * 60)
print(f"📁 Results folder : {RUNS_DIR / RUN_NAME}")
print(f"⚖️  Best model    : {best_weights}  "
      f"({'✅ exists' if best_weights.exists() else '❌ missing'})")
print(f"⚖️  Last model    : {last_weights}  "
      f"({'✅ exists' if last_weights.exists() else '❌ missing'})")
print()
print("📊 Generated files:")
for fname in [
    "results.csv", "training_curves.png",
    "map_per_class.png", "val_comparison_grid.png",
]:
    p = RUNS_DIR / RUN_NAME / fname
    print(f"   {'✅' if p.exists() else '❌'} {fname}")
print()
print("🚀 Quick inference:")
print(f"   from ultralytics import YOLO")
print(f"   model = YOLO(r'{best_weights}')")
print(f"   results = model.predict('your_image.jpg', conf=0.25, iou=0.45)")
print(f"   results[0].show()")
print()
print("💡 If mAP < 0.50 for a class → add more data or use targeted augmentation.")
print("=" * 60)
